# Results Plots:

### Author : Ricardo Queiroz (UFRN)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from itertools import cycle
import os
import re

font_dir = '/usr/share/fonts/truetype/msttcorefonts/'
fm.fontManager.addfont(font_dir + 'Times_New_Roman.ttf')
plt.rcParams['font.family'] = 'Times New Roman'

## Paths

In [2]:
# Changing Working Directory - Linux
campaigns_path = '../measurement_results/'
campaign_name = 'magazine_paper'
results_path = '../plot_results/'+campaign_name 

metrics = ['tput5g', 'tputFWA']
#metrics = ['psnr', 'ssim', 'tput5g', 'tputFWA', 'vamp']
modulation = ['64qam', '256qam']
users = [1, 3, 5]
#antenna = ['mimo']
antenna = ['siso','mimo']
njobs = 10

In [3]:
dados = {}
acumulados = {}

def parse_bitrate(val):
    try:
        val = val.strip().upper()
        if val.endswith('M'):
            return float(val.replace('M', ''))
        elif val.endswith('K'):
            return float(val.replace('K', '')) / 1000
        elif val.endswith('B'):
            return float(val.replace('B', '')) / 1_000_000
        else:
            return float(val)
    except:
        return None

for n in range(njobs):
    path = os.path.join(campaigns_path, campaign_name, f"JOB{n}/")

    for filename in os.listdir(path):
        if not filename.endswith('.csv'):
            continue

        complete_path = os.path.join(path, filename)
        var_name = os.path.splitext(filename)[0] + f"_JOB{n}"

        try:
            df_csv = pd.read_csv(complete_path)

            # Verifica se colunas esperadas existem
            if set(['Frame', 'Bitrate', 'Resolução']).issubset(df_csv.columns):
                df_csv['bitrate_Mbps'] = df_csv['Bitrate'].apply(parse_bitrate)

                df_csv.rename(columns={
                    'Frame': 'frame',
                    'Resolução': 'resolucao'
                }, inplace=True)
                
                # Extrai apenas a altura (número após o 'x') da resolução
                df_csv['resolucao'] = df_csv['resolucao'].astype(str).str.extract(r'x(\d+)', expand=False)
                df_csv['resolucao'] = pd.to_numeric(df_csv['resolucao'], errors='coerce')
                
                df = df_csv[['frame', 'bitrate_Mbps', 'resolucao']].copy()
                dados[var_name] = df
            #else:
                #print(f"Formato inesperado em {filename}, colunas ausentes.")
        except Exception as e:
            print(f"Erro ao ler {filename}: {e}")

# Acumula os dados por arquivo
for var_name, df in dados.items():
    if var_name not in acumulados:
        acumulados[var_name] = []
    acumulados[var_name].append(df)

# Concatena os dados acumulados
for var_name in acumulados:
    acumulados[var_name] = pd.concat(acumulados[var_name], ignore_index=True)

In [4]:
grupo_dfs = {
    'quality_256qam_siso_1user': [],
    'quality_256qam_siso_3user': [],
    'quality_256qam_siso_5user': [],
    'quality_64qam_siso_1user': [],
    'quality_64qam_siso_3user': [],
    'quality_64qam_siso_5user': [],

    'quality_256qam_mimo_1user': [],
    'quality_256qam_mimo_3user': [],
    'quality_256qam_mimo_5user': [],
    'quality_64qam_mimo_1user': [],
    'quality_64qam_mimo_3user': [],
    'quality_64qam_mimo_5user': [],
}

for k in acumulados:
    if re.match(r'quality_256qam_siso_1user_JOB\d+', k):
        grupo_dfs['quality_256qam_siso_1user'].append(acumulados[k])
    elif re.match(r'quality_256qam_siso_3user_JOB\d+', k):
        grupo_dfs['quality_256qam_siso_3user'].append(acumulados[k])
    elif re.match(r'quality_256qam_siso_5user_JOB\d+', k):
        grupo_dfs['quality_256qam_siso_5user'].append(acumulados[k])
    elif re.match(r'quality_64qam_siso_1user_JOB\d+', k):
        grupo_dfs['quality_64qam_siso_1user'].append(acumulados[k])
    elif re.match(r'quality_64qam_siso_3user_JOB\d+', k):
        grupo_dfs['quality_64qam_siso_3user'].append(acumulados[k])
    elif re.match(r'quality_64qam_siso_5user_JOB\d+', k):
        grupo_dfs['quality_64qam_siso_5user'].append(acumulados[k])

    elif re.match(r'quality_256qam_mimo_1user_JOB\d+', k):
        grupo_dfs['quality_256qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'quality_256qam_mimo_3user_JOB\d+', k):
        grupo_dfs['quality_256qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'quality_256qam_mimo_5user_JOB\d+', k):
        grupo_dfs['quality_256qam_mimo_5user'].append(acumulados[k])
    elif re.match(r'quality_64qam_mimo_1user_JOB\d+', k):
        grupo_dfs['quality_64qam_mimo_1user'].append(acumulados[k])
    elif re.match(r'quality_64qam_mimo_3user_JOB\d+', k):
        grupo_dfs['quality_64qam_mimo_3user'].append(acumulados[k])
    elif re.match(r'quality_64qam_mimo_5user_JOB\d+', k):
        grupo_dfs['quality_64qam_mimo_5user'].append(acumulados[k])

# Agora calcular o empilhamento e a média para cada grupo
medias_por_job = {}

for grupo, dfs in grupo_dfs.items():
    if dfs:  # evita erro se a lista estiver vazia
        medias_por_job[grupo] = pd.concat(dfs, ignore_index=False)

/tmp/ipykernel_5160/668988248.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  medias_por_job[grupo] = pd.concat(dfs, ignore_index=False)


## Plots

In [5]:
fontSize = 24
fontSizeLeg = 20
figsize=(10,8)
markers = ['o', 's', '^', 'D', 'v', '*', 'x', 'P', 'H', '<', '>']
hatch=['\\\\','x','O']
cmap = plt.get_cmap("tab10")
plt.rc('font', size=fontSize)

### QUALITY

In [9]:
for mode in antenna:
    for modulations in modulation:

        # Monta uma lista dos usuários que possuem dados para esta configuração
        usuarios_validos = [user for user in users if f'quality_{modulations}_{mode}_{user}user' in medias_por_job]

        if not usuarios_validos:
            print(f"Nenhum dado disponível para modulação={modulations}, modo={mode}")
            continue

        # Dicionário para armazenar as contagens de resolução por usuário
        res_counts_dict = {}

        # Coleta contagens para cada usuário
        for user in usuarios_validos:
            key = f'quality_{modulations}_{mode}_{user}user'
            res_counts = medias_por_job[key]['resolucao'].value_counts()
            res_counts_dict[user] = res_counts

        # Descobre todas as resoluções únicas para alinhar o eixo X
        todas_resolucoes = sorted(set().union(*[counts.index for counts in res_counts_dict.values()]))

        # Cria um DataFrame para facilitar plotagem e alinhamento
        import pandas as pd
        df_counts = pd.DataFrame(index=todas_resolucoes)

        for user, counts in res_counts_dict.items():
            # Reindexa para garantir todas resoluções com 0 onde não tem dados
            df_counts[user] = counts.reindex(todas_resolucoes, fill_value=0)

        # Preparar o gráfico agrupado
        largura = 0.9 / len(usuarios_validos)  # largura das barras dividida pelo número de usuários
        ind = np.arange(len(todas_resolucoes))  # posições no eixo X

        plt.figure(figsize=figsize)

        cores = cmap

        for i, user in enumerate(usuarios_validos):
            plt.bar(ind + i*largura, df_counts[user], largura, label = f'{user} User' if user == 1 else f'{user} Users', color=cores(i), alpha=0.8, edgecolor='black', hatch=hatch[i])

            for x, y in zip(ind + i*largura, df_counts[user]):
                plt.text(
                    x, y + max(df_counts.max()) * 0.02 if y > 0 else max(df_counts.max()) * 0.01,
                    f'{y/1000:.1f}k' if y > 0 else '0.0',
                    ha='center', va='bottom',
                    fontsize=fontSize,
                    rotation = 90,
                    color=cores(i)
                )
        
        plt.xlabel('Video Quality', fontsize=fontSize)
        plt.ylabel('Frequency (frames)', fontsize=fontSize)
        #plt.title(f'Quality - {mode} - {modulations}')
        label_map = {
            360: 'Low',
            480: 'SD',
            720: 'HD',
            1080: 'FHD',
            2160: '4K'
        }
        
        # Gera os rótulos customizados para o eixo X
        xtick_labels = [label_map.get(r, str(r)) for r in todas_resolucoes]
        
        plt.xticks(ind + largura*(len(usuarios_validos)-1)/2, xtick_labels)
        plt.ylim (0,130000)
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()

        # Salvar figura
        os.makedirs(results_path, exist_ok=True)
        complete_path = os.path.join(results_path, f"quality_{modulations}_{mode}.png")
        plt.savefig(complete_path, dpi=600, bbox_inches='tight', transparent=True)
        plt.close()
